# Show-o2: text-derived red steering

**Can a text-derived red steering signal increase red elements in generated images?**

Select a GPU runtime (A100 / High RAM when available). Run setup once, restart
the session, then load the model. Push this branch before using it in Colab.

Baseline, aligned text-vector extraction, and front injection are implemented.
Real Show-o2 GPU execution remains unverified. The general demo stays in `colab.ipynb`.


In [ ]:
import subprocess

subprocess.run(["nvidia-smi"], check=True)


## 1. Setup

Run once in a fresh Colab runtime. Use our experiment branch and a fixed upstream commit.


In [ ]:
import subprocess

branch = "experiment/text-red-steering"
commit = "45a5a2de01d1ebd10cd5864d29310a76476cdf23"

subprocess.run(["git", "clone", "--branch", branch,
                "https://github.com/Nicerova7/show-o2-space.git",
                "/content/show-o2-space"], check=True)
subprocess.run(["git", "clone", "https://github.com/showlab/Show-o.git",
                "/content/Show-o"], check=True)
subprocess.run(["git", "-C", "/content/Show-o", "checkout", commit], check=True)


In [ ]:
# Keep Colab's matched PyTorch and torchvision installation.
get_ipython().run_line_magic("pip", 'install transformers==4.47.0 diffusers==0.31.0 huggingface-hub==0.27.1 accelerate==1.2.1 timm==1.0.12 einops==0.8.0 torchdiffeq==0.2.5 "numpy>=1.26,<3" "Pillow>=10.4,<12" sentencepiece==0.2.1')


### Load the model

Select **Runtime > Restart session**, then continue here. Run model loading once;
reuse `adapter` for the following experiments.


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

import torch
from IPython.display import display

PROJECT = Path("/content/show-o2-space")
UPSTREAM = Path("/content/Show-o")
os.chdir(PROJECT)
sys.path[:0] = [str(PROJECT), str(UPSTREAM / "show-o2")]

from show_o2_space.adapter import MODEL_ID, ShowO2Adapter

adapter = ShowO2Adapter()
print(torch.cuda.get_device_name(0), adapter.dtype)


## 2. Generation defaults and baseline

Edit `default_kwargs` to control Show-o2 generation. `model` is our loaded adapter;
`steps` controls flow sampling, `guidance` controls CFG, and `seed` fixes the initial noise.
Run this cell again when changing defaults, then regenerate the baseline.


In [ ]:
model = adapter
default_kwargs = dict(model=model, steps=50, guidance=5.0, seed=42)

def generate_image(prompt, model, steps, guidance, seed):
    return model.generate(prompt, steps=steps, guidance=guidance, seed=seed)

prompt = "A photograph of a car parked on a street."
output = PROJECT / "outputs/steering-red"
output.mkdir(parents=True, exist_ok=True)


In [ ]:
baseline = generate_image(prompt, **default_kwargs)
baseline.save(output / "baseline.png")
metadata = dict(
    model=MODEL_ID, prompt=prompt,
    **{key: default_kwargs[key] for key in ("steps", "guidance", "seed")},
    dtype=str(model.dtype), stage="baseline_only",
    upstream_revision=subprocess.check_output(
        ["git", "-C", str(UPSTREAM), "rev-parse", "HEAD"], text=True
    ).strip(),
    project_revision=subprocess.check_output(
        ["git", "-C", str(PROJECT), "rev-parse", "HEAD"], text=True
    ).strip(),
)
(output / "baseline.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
display(baseline)


## 3. ActAdd: red minus whitespace

Use the Show-o2 backbone with `positive_text="red"`, `negative_text=" "`, and `c=5`.
Prepend the same BOS, then right-pad the shorter token sequence with space tokens,
as in ActAdd. The cell prints token IDs and added padding; equal lengths add nothing.
Space tokens remain visible to causal attention, rather than being masked as PAD.

Capture the residual stream **entering** decoder block `layer` (zero-based), then
subtract every aligned position. No averaging, last-token selection, or normalization.
The result has shape `[1, aligned_sequence_length, hidden_dimension]`.


In [ ]:
from experiments.steering_utils import align_prompt_tokens, capture_layer_input

@torch.no_grad()
def extract_text_hidden(token_ids, model, layer):
    input_ids = torch.tensor([token_ids], device=model.device)
    length = input_ids.shape[1]
    causal_mask = torch.full(
        (length, length), float("-inf"), device=model.device, dtype=model.dtype
    ).triu(diagonal=1)[None, None]

    with capture_layer_input(model.layers, layer) as captured, model.autocast():
        model.model.showo.model(
            input_ids=input_ids, attention_mask=causal_mask,
            use_cache=False, output_hidden_states=False, return_dict=True,
        )
    return captured["hidden"]


In [ ]:
steering_kwargs = dict(
    layer=8,
    c=5.0,
    positive_text="red",
    negative_text=" ",
)
print("Available layers:", list(range(len(model.layers))))


In [ ]:
# Rerun after changing layer or either text.
layer = steering_kwargs["layer"]
red_ids, control_ids, padding = align_prompt_tokens(
    steering_kwargs["positive_text"], steering_kwargs["negative_text"],
    tokenizer=model.tokenizer, bos_id=model.token_ids["bos_id"],
)
print("red IDs / tokens:", red_ids, model.tokenizer.convert_ids_to_tokens(red_ids))
print("space IDs / tokens:", control_ids, model.tokenizer.convert_ids_to_tokens(control_ids))
print("Space tokens added on the right:", padding)

hidden_red = extract_text_hidden(red_ids, model=model, layer=layer)
hidden_control = extract_text_hidden(control_ids, model=model, layer=layer)
print("Aligned states:", hidden_red.shape, hidden_control.shape)

extraction_metadata = dict(
    layer=layer, positive_text=steering_kwargs["positive_text"],
    negative_text=steering_kwargs["negative_text"],
    red_token_ids=red_ids, control_token_ids=control_ids, padding=padding,
    capture="decoder block input (resid_pre)", alignment="tokens_right, space padding",
)


In [ ]:
# Rerun after changing c. Keep all aligned sequence positions.
c = steering_kwargs["c"]
h_red = hidden_red.float()
h_control = hidden_control.float()
v_red = h_red - h_control
steering_vector = c * v_red

print("h_red / h_control / v_red:", h_red.shape, h_control.shape, v_red.shape)
print("Difference norm at each token position:", v_red.norm(dim=-1))
print("c:", c, "Scaled vector norm:", steering_vector.norm().item())


`v_red` is the raw red-minus-whitespace activation difference, including BOS.
The common BOS should cancel under causal attention. `steering_vector` is already
scaled by `c`; section 4 injects it directly, without multiplying again.


In [ ]:
torch.save(
    dict(h_red=h_red, h_control=h_control, v_red=v_red,
         steering_vector=steering_vector,
         metadata=dict(**extraction_metadata, c=c, model=MODEL_ID)),
    output / "text_direction.pt",
)
print("Saved:", output / "text_direction.pt")


## 4. Generate with front injection

Inject from position 0 at the input of the extraction layer, only in the prompt
row. BOS cancels; for red versus space, the change lands on "A".

`steering_steps`: `0` = off, `12` = first 12 evaluations, `24` = first 24,
`"last"` = last evaluation only, `"all"` = every evaluation.
An integer larger than the available evaluations also applies throughout.
The zero-vector run below measures the actual number of evaluations.

Keep the baseline's prompt, seed, steps, and guidance. Rerun section 3 when
changing layer or texts; rerun scaling when changing `c`.


In [ ]:
from experiments.steering_utils import inject_layer_input

# Keep the whole aligned vector inside BOS + prompt, before image positions.
prompt_ids = model.tokenizer(prompt, add_special_tokens=False)["input_ids"][:290]
assert steering_vector.shape[1] <= 1 + len(prompt_ids), "Use a prompt at least as long as the steering pair."
injection_layer = extraction_metadata["layer"]


### Check c=0

A zero-vector intervention should reproduce the baseline with identical settings.


In [ ]:
with inject_layer_input(model.layers, injection_layer, torch.zeros_like(steering_vector)) as zero_run:
    zero_image = generate_image(prompt, **default_kwargs)

assert zero_image.tobytes() == baseline.tobytes(), "c=0 differs: rerun baseline with the same defaults."
print("c=0 matches baseline. Flow evaluations:", zero_run["calls"])


### Compare injection timing

Edit the list to choose cases. `"last"` means one injection at the end, not all
steps up to the end. Use `["all"]` to repeat the previous experiment.


In [ ]:
steering_steps_cases = [0, 12, 24, "last"]
results = {}

for steering_steps in steering_steps_cases:
    with inject_layer_input(
        model.layers, injection_layer, steering_vector,
        steering_steps=steering_steps, total_steps=zero_run["calls"],
    ) as run:
        image = generate_image(prompt, **default_kwargs)

    if steering_steps == 0:
        assert image.tobytes() == baseline.tobytes(), "0 injections must match baseline."
    image.save(output / f"steered_{steering_steps}.png")
    run_metadata = dict(
        model=MODEL_ID, prompt=prompt,
        **{key: default_kwargs[key] for key in ("steps", "guidance", "seed")},
        **extraction_metadata, c=c, start_position=0, steering_steps=steering_steps,
        flow_evaluations=run["calls"], injections=run["injections"],
        injection="block input; conditional row only",
        upstream_revision=metadata["upstream_revision"],
        project_revision=metadata["project_revision"], dtype=str(model.dtype),
    )
    (output / f"steered_{steering_steps}.json").write_text(json.dumps(run_metadata, indent=2), encoding="utf-8")
    results[steering_steps] = image
    print(f"{steering_steps}: {run['injections']} injections / {run['calls']} evaluations")


In [ ]:
from PIL import Image, ImageDraw

comparison = Image.new("RGB", (baseline.width * len(results), baseline.height + 28), "white")
labels = ImageDraw.Draw(comparison)
for column, (steering_steps, image) in enumerate(results.items()):
    x = column * baseline.width
    labels.text((x + 8, 8), f"steering_steps={steering_steps}", fill="black")
    comparison.paste(image, (x, 28))
comparison.save(output / "comparison.png")
display(comparison)


## 5. Compare and evaluate (implementation pending)

Display baseline and steered images side by side with identical prompts and seeds.
Report red pixel coverage as a simple proxy, alongside object preservation and
image quality: an overall red tint alone is not evidence of useful attribute control.
Choose layer/intensity on validation prompts, then evaluate held-out prompts and
multiple seeds. Save vectors, positions, layer, coefficients, sampling settings,
and model revision with the results. Pin checkpoint revisions before the study.
